# Cross-Model ATR: Pythia-410m — Attractor Dominance & Basin Mapping

## Experiment Design
**This notebook is an EXACT replication of the GPT-2 Small ATR experiment (EXP_009d1),**
**applied to EleutherAI's Pythia-410m. Fourth corner of the 2×2 factorial design.**

### 2×2 Design Context
| | WebText (Reddit) | The Pile (diverse) |
| --- | --- | --- |
| **Small** (~100M, d=768) | GPT-2 Small ✅ | Pythia-160m |
| **Medium** (~300M, d=1024) | GPT-2 Medium | **Pythia-410m** ← this |

### Key Difference: Scale + Different Training Data
- **Pythia-160m:** 85M params, 12 layers, d_model=768, The Pile
- **Pythia-410m:** 302M params, 24 layers, d_model=1024, The Pile

### Hypotheses Under Test

**H_4C1: Scale Replication** — Does basin count scale with capacity on The Pile,
replicating the GPT-2 Small → Medium scaling result?

**H_4C2: Corpus Replication** — Do Pythia-410m basins match GPT-2 Medium basins
(same scale, different data), or Pythia-160m basins (same data, different scale)?

**H_4C3: Interaction Effect** — Is the attractor landscape driven primarily by
training data, model scale, or their interaction?

### Method
Identical to EXP_009d1. Same 125 prompts, same iteration schedule, same ATR engine.
Only the model loaded in STEP 1 is changed.

---


In [1]:
# ============================================================
# STEP 0: DEPENDENCIES
# ============================================================
import sys
!{sys.executable} -m pip install kaleido -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("pythia-410m", device=device)
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")
print(f"Running on: {device}")

# Output directory for all saved artifacts
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model pythia-410m into HookedTransformer
Architecture: 24 layers, 16 heads, d_model=1024
Running on: cpu
Output directory: c:\Users\Fab2\Desktop\AI\_learn\_fold\03_MECHINIPHYLUM\_LAB_NOTEBOOKS\lucier-repo\experiments\pythia_410m\output


In [14]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', '..')))

# ============================================================
# STEP 2: CONFIGURATION — 125 Prompts from prompt_library.py
# ============================================================
from prompt_library import (
    PROMPT_LIBRARY, PREDICTIONS, CATEGORY_MAP,
    COMPLEX, NARRATIVE, SIMPLE, CHEMICAL, ACRONYMS, VULGARITY, WILD
)

# Tightened schedule: convergence occurs by ~100
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100, 150, 200, 250]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} -> {LAYER_END}")
print(f"Total prompts: {len(PROMPT_LIBRARY)}")
print(f"\nBreakdown:")
for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    print(f"  {cat_name}: {len(cat_dict)} prompts")

# Save config
config_md = f"""# Stage 1 Run Config\n
- **Prompts:** {len(PROMPT_LIBRARY)}\n
- **Schedule:** {ITERATION_SCHEDULE}\n
- **Layers:** {LAYER_START} -> {LAYER_END}\n
- **Device:** {device}\n
"""
with open(os.path.join(OUTPUT_DIR, 'config.md'), 'w', encoding='utf-8') as f:
    f.write(config_md)
print(f"\n[SAVED] {OUTPUT_DIR}/config.md")

Schedule: [0, 2, 3, 5, 10, 20, 50, 100, 150, 200, 250]
Room: Layers 0 -> 23
Total prompts: 125

Breakdown:
  Complex: 25 prompts
  Narrative: 20 prompts
  Simple: 20 prompts
  Chemical: 10 prompts
  Acronyms: 10 prompts
  Vulgarity: 10 prompts
  Wild: 30 prompts

[SAVED] output/config.md


In [16]:
# ============================================================
# STEP 3: THE CORE ENGINE — Identical to lucier_total_resonance
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """
    TOTAL Lucier Loop: iteratively re-inject the ENTIRE residual stream
    tensor (all token positions) through the layer slice.
    Returns a list of snapshot dicts at each scheduled iteration.
    """
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        # Normalise to maintain energy level
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>3}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.4f}, pos_collapse={position_similarity:.4f}")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Engine loaded.")

Engine loaded.


In [17]:
# ============================================================
# STEP 4: RUN ALL 125 PROMPTS
# ============================================================

all_results = {}

for idx, (label, prompt) in enumerate(PROMPT_LIBRARY.items()):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(PROMPT_LIBRARY)}] RECORDING: '{label}'")
    print(f"  Prompt: \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    print(f"  y Terminal token: '{terminal}'")

print(f"\n{'='*60}")
print(f"ALL {len(all_results)} RECORDINGS COMPLETE.")


[1/125] RECORDING: 'A01_physics'
  Prompt: "The implications of quantum entanglement suggest that"
  iter   2: top='s', cos_mean=0.8172, pos_collapse=0.4213
  iter   3: top='el', cos_mean=0.9448, pos_collapse=0.5565
  iter   5: top='', cos_mean=0.9337, pos_collapse=0.5769
  iter  10: top='en', cos_mean=0.8445, pos_collapse=0.6567
  iter  20: top='p', cos_mean=0.7519, pos_collapse=0.9930
  iter  50: top='D', cos_mean=0.9161, pos_collapse=1.0000
  iter 100: top='.', cos_mean=0.4716, pos_collapse=1.0000
  iter 150: top='OF', cos_mean=0.9451, pos_collapse=0.5589
  iter 200: top='t', cos_mean=0.9322, pos_collapse=0.5845
  iter 250: top='S', cos_mean=0.8897, pos_collapse=0.4036
  y Terminal token: 'S'

[2/125] RECORDING: 'A02_medical'
  Prompt: "A meta-analysis of randomised controlled trials indicates"
  iter   2: top='s', cos_mean=0.7804, pos_collapse=0.4826
  iter   3: top='s', cos_mean=0.8823, pos_collapse=0.5689
  iter   5: top='ics', cos_mean=0.9494, pos_collapse=0.5808
  iter  10: to

---
## 5. Analysis

### 5a. Hypothesis Assessment — Predictions vs Actuals

In [18]:
# ============================================================
# VIS 5a: HYPOTHESIS ASSESSMENT — Predictions vs Actuals
# ============================================================

md = "# Stage 1 Results: Hypothesis Assessment\n\n"
md += "| Prompt | Category | Predicted | Actual Terminal | Match? |\n"
md += "|:---|:---|:---|:---|:---|\n"

basin_counts = {}
category_basins = {}
mismatches = []

for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    predicted_basin, confidence = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    
    # Classify actual basin
    # AFTER (exact match)
    if terminal == 'prolet':
        actual_basin = 'prolet'
    elif terminal == 'Divine':
        actual_basin = 'Divine'
    else:
        actual_basin = f'OTHER:{terminal}'
    
    basin_counts[actual_basin] = basin_counts.get(actual_basin, 0) + 1
    
    if category not in category_basins:
        category_basins[category] = []
    category_basins[category].append((label, actual_basin, terminal))
    
    match = 'y' if predicted_basin.lower() in actual_basin.lower() else 'n'
    if match == 'n' and predicted_basin != 'unknown':
        mismatches.append((label, predicted_basin, actual_basin))
    
    md += f"| {label} | {category} | `{predicted_basin}` ({confidence}) | `{terminal}` -> **{actual_basin}** | {match} |\n"

md += "\n---\n\n"
md += "## Basin Summary\n\n"
md += "| Basin | Count | % |\n"
md += "|:---|:---|:---|\n"
total = len(all_results)
for basin, count in sorted(basin_counts.items(), key=lambda x: -x[1]):
    md += f"| **{basin}** | {count} | {count/total*100:.1f}% |\n"

md += "\n---\n\n"
md += "## Category Breakdown\n\n"
for cat, entries in category_basins.items():
    md += f"### {cat} ({len(entries)} prompts)\n"
    cat_basins = {}
    for label, basin, tok in entries:
        cat_basins[basin] = cat_basins.get(basin, 0) + 1
    for b, c in sorted(cat_basins.items(), key=lambda x: -x[1]):
        md += f"- {b}: {c}/{len(entries)}\n"
    md += "\n"

if mismatches:
    md += "## Prediction Mismatches\n\n"
    for label, pred, actual in mismatches:
        md += f"- **{label}**: predicted `{pred}`, got `{actual}`\n"

# Save and display
with open(os.path.join(OUTPUT_DIR, 'hypothesis_assessment.md'), 'w') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/hypothesis_assessment.md")
display(Markdown(md))

[SAVED] output/hypothesis_assessment.md


# Stage 1 Results: Hypothesis Assessment

| Prompt | Category | Predicted | Actual Terminal | Match? |
|:---|:---|:---|:---|:---|
| A01_physics | Complex | `prolet` (high) | `S` -> **OTHER:S** | n |
| A02_medical | Complex | `prolet` (high) | `just` -> **OTHER:just** | n |
| A03_neuro | Complex | `prolet` (high) | `f` -> **OTHER:f** | n |
| A04_climate | Complex | `prolet` (high) | `/` -> **OTHER:/** | n |
| A05_evolution | Complex | `prolet` (high) | `on` -> **OTHER:on** | n |
| A06_epistemology | Complex | `prolet` (high) | `.` -> **OTHER:.** | n |
| A07_sociology | Complex | `prolet` (high) | `` -> **OTHER:** | n |
| A08_linguistics | Complex | `prolet` (high) | `T` -> **OTHER:T** | n |
| A09_code | Complex | `prolet` (high) | `.` -> **OTHER:.** | n |
| A10_sql | Complex | `prolet` (high) | `` -> **OTHER:** | n |
| A11_ml | Complex | `prolet` (high) | `/` -> **OTHER:/** | n |
| A12_systems | Complex | `prolet` (high) | `` -> **OTHER:** | n |
| A13_networking | Complex | `prolet` (high) | `/` -> **OTHER:/** | n |
| A14_kant | Complex | `prolet` (high) | `four` -> **OTHER:four** | n |
| A15_sartre | Complex | `prolet` (high) | `and` -> **OTHER:and** | n |
| A16_wittgenstein | Complex | `prolet` (high) | `Fil` -> **OTHER:Fil** | n |
| A17_marx | Complex | `prolet` (high) | `.` -> **OTHER:.** | n |
| A18_gothic | Complex | `prolet` (high) | `` -> **OTHER:** | n |
| A19_romantic | Complex | `prolet` (high) | `` -> **OTHER:** | n |
| A20_modernist | Complex | `prolet` (high) | `/` -> **OTHER:/** | n |
| A21_dickens | Complex | `prolet` (high) | `.` -> **OTHER:.** | n |
| A22_legal | Complex | `prolet` (high) | `en` -> **OTHER:en** | n |
| A23_contract | Complex | `prolet` (high) | `ff` -> **OTHER:ff** | n |
| A24_patent | Complex | `prolet` (high) | `` -> **OTHER:** | n |
| A25_academic_abs | Complex | `prolet` (high) | `"` -> **OTHER:"** | n |
| B01_napoleon | Narrative | `prolet` (medium) | `/` -> **OTHER:/** | n |
| B02_wwi | Narrative | `prolet` (medium) | `...` -> **OTHER:...** | n |
| B03_moon | Narrative | `prolet` (medium) | `ed` -> **OTHER:ed** | n |
| B04_rome | Narrative | `prolet` (medium) | `/` -> **OTHER:/** | n |
| B05_mlk | Narrative | `prolet` (medium) | `.` -> **OTHER:.** | n |
| B06_sources | Narrative | `prolet` (medium) | `of` -> **OTHER:of** | n |
| B07_breaking | Narrative | `prolet` (medium) | `of` -> **OTHER:of** | n |
| B08_editorial | Narrative | `prolet` (medium) | `` -> **OTHER:** | n |
| B09_sports | Narrative | `prolet` (medium) | `and` -> **OTHER:and** | n |
| B10_weather | Narrative | `prolet` (medium) | `AND` -> **OTHER:AND** | n |
| B11_alone | Narrative | `prolet` (medium) | `UN` -> **OTHER:UN** | n |
| B12_fear | Narrative | `prolet` (medium) | `T` -> **OTHER:T** | n |
| B13_joy | Narrative | `prolet` (medium) | `of` -> **OTHER:of** | n |
| B14_anger | Narrative | `prolet` (medium) | `` -> **OTHER:** | n |
| B15_casual | Narrative | `prolet` (medium) | `till` -> **OTHER:till** | n |
| B16_gossip | Narrative | `prolet` (medium) | `F` -> **OTHER:F** | n |
| B17_argument | Narrative | `prolet` (medium) | `en` -> **OTHER:en** | n |
| B18_advice | Narrative | `prolet` (medium) | `AND` -> **OTHER:AND** | n |
| B19_question | Narrative | `prolet` (medium) | `of` -> **OTHER:of** | n |
| B20_reddit | Narrative | `prolet` (medium) | `or` -> **OTHER:or** | n |
| D01_water | Chemical | `unknown` (none) | `o` -> **OTHER:o** | n |
| D02_periodic | Chemical | `unknown` (none) | `.` -> **OTHER:.** | n |
| D03_organic | Chemical | `unknown` (none) | `/` -> **OTHER:/** | n |
| D04_equation | Chemical | `unknown` (none) | `Lif` -> **OTHER:Lif** | n |
| D05_amino | Chemical | `unknown` (none) | `/` -> **OTHER:/** | n |
| D06_physics_eq | Chemical | `unknown` (none) | `lee` -> **OTHER:lee** | n |
| D07_dna | Chemical | `unknown` (none) | `v` -> **OTHER:v** | n |
| D08_math | Chemical | `unknown` (none) | `es` -> **OTHER:es** | n |
| D09_units | Chemical | `unknown` (none) | `es` -> **OTHER:es** | n |
| D10_isotopes | Chemical | `unknown` (none) | `` -> **OTHER:** | n |
| E01_politics | Acronyms | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| E02_tech | Acronyms | `unknown` (none) | `/` -> **OTHER:/** | n |
| E03_orgs | Acronyms | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| E04_internet | Acronyms | `unknown` (none) | `.` -> **OTHER:.** | n |
| E05_finance | Acronyms | `unknown` (none) | `and` -> **OTHER:and** | n |
| E06_medical | Acronyms | `unknown` (none) | `and` -> **OTHER:and** | n |
| E07_military | Acronyms | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| E08_academic | Acronyms | `unknown` (none) | `/` -> **OTHER:/** | n |
| E09_mixed | Acronyms | `unknown` (none) | `T` -> **OTHER:T** | n |
| E10_crypto | Acronyms | `unknown` (none) | `end` -> **OTHER:end** | n |
| C01_jack_jill | Simple | `Divine` (low) | `of` -> **OTHER:of** | n |
| C02_king_cole | Simple | `Divine` (low) | `OF` -> **OTHER:OF** | n |
| C03_mary_lamb | Simple | `Divine` (low) | `wit` -> **OTHER:wit** | n |
| C04_humpty | Simple | `Divine` (low) | `.` -> **OTHER:.** | n |
| C05_twinkle | Simple | `Divine` (low) | `PARAM` -> **OTHER:PARAM** | n |
| C06_dog | Simple | `Divine` (low) | `and` -> **OTHER:and** | n |
| C07_cat_mat | Simple | `Divine` (low) | `"` -> **OTHER:"** | n |
| C08_boy_girl | Simple | `Divine` (low) | `y` -> **OTHER:y** | n |
| C09_run | Simple | `Divine` (low) | `ty` -> **OTHER:ty** | n |
| C10_spot | Simple | `Divine` (low) | `!"` -> **OTHER:!"** | n |
| C11_genesis | Simple | `Divine` (low) | `.` -> **OTHER:.** | n |
| C12_beatitudes | Simple | `Divine` (low) | `en` -> **OTHER:en** | n |
| C13_psalm | Simple | `Divine` (low) | `f` -> **OTHER:f** | n |
| C14_commandment | Simple | `Divine` (low) | `/` -> **OTHER:/** | n |
| C15_fox_hen | Simple | `Divine` (low) | `/` -> **OTHER:/** | n |
| C16_ant_dove | Simple | `Divine` (low) | `f` -> **OTHER:f** | n |
| C17_tortoise | Simple | `Divine` (low) | `` -> **OTHER:** | n |
| C18_wolf | Simple | `Divine` (low) | `` -> **OTHER:** | n |
| C19_lion_mouse | Simple | `Divine` (low) | `as` -> **OTHER:as** | n |
| C20_crow | Simple | `Divine` (low) | `o` -> **OTHER:o** | n |
| F01_anger | Vulgarity | `unknown` (none) | `.` -> **OTHER:.** | n |
| F02_insult | Vulgarity | `unknown` (none) | `ATT` -> **OTHER:ATT** | n |
| F03_frustration | Vulgarity | `unknown` (none) | `f` -> **OTHER:f** | n |
| F04_argument | Vulgarity | `unknown` (none) | `` -> **OTHER:** | n |
| F05_rant | Vulgarity | `unknown` (none) | `or` -> **OTHER:or** | n |
| F06_dismissal | Vulgarity | `unknown` (none) | `or` -> **OTHER:or** | n |
| F07_shock | Vulgarity | `unknown` (none) | `not` -> **OTHER:not** | n |
| F08_mild | Vulgarity | `unknown` (none) | `o` -> **OTHER:o** | n |
| F09_slur_adjacent | Vulgarity | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| F10_exasperation | Vulgarity | `unknown` (none) | `` -> **OTHER:** | n |
| G01_punctuation | Wild | `unknown` (none) | `three` -> **OTHER:three** | n |
| G02_brackets | Wild | `unknown` (none) | `."` -> **OTHER:."** | n |
| G03_counting | Wild | `unknown` (none) | `en` -> **OTHER:en** | n |
| G04_fibonacci | Wild | `unknown` (none) | `.` -> **OTHER:.** | n |
| G05_primes | Wild | `unknown` (none) | `` -> **OTHER:** | n |
| G06_binary | Wild | `unknown` (none) | `and` -> **OTHER:and** | n |
| G07_the | Wild | `unknown` (none) | `DI` -> **OTHER:DI** | n |
| G08_period | Wild | `unknown` (none) | `Apart` -> **OTHER:Apart** | n |
| G09_space | Wild | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| G10_newline | Wild | `unknown` (none) | `f` -> **OTHER:f** | n |
| G11_aaa | Wild | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| G12_the_the | Wild | `unknown` (none) | `ple` -> **OTHER:ple** | n |
| G13_buffalo | Wild | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| G14_nursery_acad | Wild | `unknown` (none) | `.` -> **OTHER:.** | n |
| G15_bible_code | Wild | `unknown` (none) | `-` -> **OTHER:-** | n |
| G16_nursery_vulgar | Wild | `unknown` (none) | `` -> **OTHER:** | n |
| G17_formal_slang | Wild | `unknown` (none) | `/` -> **OTHER:/** | n |
| G18_french | Wild | `unknown` (none) | `/` -> **OTHER:/** | n |
| G19_german | Wild | `unknown` (none) | `FI` -> **OTHER:FI** | n |
| G20_spanish | Wild | `unknown` (none) | `/` -> **OTHER:/** | n |
| G21_latin | Wild | `unknown` (none) | `.` -> **OTHER:.** | n |
| G22_japanese_rom | Wild | `unknown` (none) | `` -> **OTHER:** | n |
| G23_emoji | Wild | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| G24_beatles | Wild | `unknown` (none) | `.` -> **OTHER:.** | n |
| G25_rickroll | Wild | `unknown` (none) | `s` -> **OTHER:s** | n |
| G26_bohemian | Wild | `unknown` (none) | `AND` -> **OTHER:AND** | n |
| G27_ignore | Wild | `unknown` (none) | `/` -> **OTHER:/** | n |
| G28_system | Wild | `unknown` (none) | `."` -> **OTHER:."** | n |
| G29_palindrome | Wild | `unknown` (none) | `f` -> **OTHER:f** | n |
| G30_alphabet | Wild | `unknown` (none) | `` -> **OTHER:** | n |

---

## Basin Summary

| Basin | Count | % |
|:---|:---|:---|
| **OTHER:** | 17 | 13.6% |
| **OTHER:/** | 16 | 12.8% |
| **OTHER:.** | 14 | 11.2% |
| **OTHER:AND** | 11 | 8.8% |
| **OTHER:f** | 6 | 4.8% |
| **OTHER:and** | 6 | 4.8% |
| **OTHER:of** | 5 | 4.0% |
| **OTHER:en** | 4 | 3.2% |
| **OTHER:T** | 3 | 2.4% |
| **OTHER:or** | 3 | 2.4% |
| **OTHER:o** | 3 | 2.4% |
| **OTHER:"** | 2 | 1.6% |
| **OTHER:es** | 2 | 1.6% |
| **OTHER:."** | 2 | 1.6% |
| **OTHER:S** | 1 | 0.8% |
| **OTHER:just** | 1 | 0.8% |
| **OTHER:on** | 1 | 0.8% |
| **OTHER:four** | 1 | 0.8% |
| **OTHER:Fil** | 1 | 0.8% |
| **OTHER:ff** | 1 | 0.8% |
| **OTHER:...** | 1 | 0.8% |
| **OTHER:ed** | 1 | 0.8% |
| **OTHER:UN** | 1 | 0.8% |
| **OTHER:till** | 1 | 0.8% |
| **OTHER:F** | 1 | 0.8% |
| **OTHER:Lif** | 1 | 0.8% |
| **OTHER:lee** | 1 | 0.8% |
| **OTHER:v** | 1 | 0.8% |
| **OTHER:end** | 1 | 0.8% |
| **OTHER:OF** | 1 | 0.8% |
| **OTHER:wit** | 1 | 0.8% |
| **OTHER:PARAM** | 1 | 0.8% |
| **OTHER:y** | 1 | 0.8% |
| **OTHER:ty** | 1 | 0.8% |
| **OTHER:!"** | 1 | 0.8% |
| **OTHER:as** | 1 | 0.8% |
| **OTHER:ATT** | 1 | 0.8% |
| **OTHER:not** | 1 | 0.8% |
| **OTHER:three** | 1 | 0.8% |
| **OTHER:DI** | 1 | 0.8% |
| **OTHER:Apart** | 1 | 0.8% |
| **OTHER:ple** | 1 | 0.8% |
| **OTHER:-** | 1 | 0.8% |
| **OTHER:FI** | 1 | 0.8% |
| **OTHER:s** | 1 | 0.8% |

---

## Category Breakdown

### Complex (25 prompts)
- OTHER:: 6/25
- OTHER:/: 4/25
- OTHER:.: 4/25
- OTHER:S: 1/25
- OTHER:just: 1/25
- OTHER:f: 1/25
- OTHER:on: 1/25
- OTHER:T: 1/25
- OTHER:four: 1/25
- OTHER:and: 1/25
- OTHER:Fil: 1/25
- OTHER:en: 1/25
- OTHER:ff: 1/25
- OTHER:": 1/25

### Narrative (20 prompts)
- OTHER:of: 4/20
- OTHER:/: 2/20
- OTHER:: 2/20
- OTHER:AND: 2/20
- OTHER:...: 1/20
- OTHER:ed: 1/20
- OTHER:.: 1/20
- OTHER:and: 1/20
- OTHER:UN: 1/20
- OTHER:T: 1/20
- OTHER:till: 1/20
- OTHER:F: 1/20
- OTHER:en: 1/20
- OTHER:or: 1/20

### Chemical (10 prompts)
- OTHER:/: 2/10
- OTHER:es: 2/10
- OTHER:o: 1/10
- OTHER:.: 1/10
- OTHER:Lif: 1/10
- OTHER:lee: 1/10
- OTHER:v: 1/10
- OTHER:: 1/10

### Acronyms (10 prompts)
- OTHER:AND: 3/10
- OTHER:/: 2/10
- OTHER:and: 2/10
- OTHER:.: 1/10
- OTHER:T: 1/10
- OTHER:end: 1/10

### Simple (20 prompts)
- OTHER:.: 2/20
- OTHER:f: 2/20
- OTHER:/: 2/20
- OTHER:: 2/20
- OTHER:of: 1/20
- OTHER:OF: 1/20
- OTHER:wit: 1/20
- OTHER:PARAM: 1/20
- OTHER:and: 1/20
- OTHER:": 1/20
- OTHER:y: 1/20
- OTHER:ty: 1/20
- OTHER:!": 1/20
- OTHER:en: 1/20
- OTHER:as: 1/20
- OTHER:o: 1/20

### Vulgarity (10 prompts)
- OTHER:: 2/10
- OTHER:or: 2/10
- OTHER:.: 1/10
- OTHER:ATT: 1/10
- OTHER:f: 1/10
- OTHER:not: 1/10
- OTHER:o: 1/10
- OTHER:AND: 1/10

### Wild (30 prompts)
- OTHER:AND: 5/30
- OTHER:.: 4/30
- OTHER:: 4/30
- OTHER:/: 4/30
- OTHER:.": 2/30
- OTHER:f: 2/30
- OTHER:three: 1/30
- OTHER:en: 1/30
- OTHER:and: 1/30
- OTHER:DI: 1/30
- OTHER:Apart: 1/30
- OTHER:ple: 1/30
- OTHER:-: 1/30
- OTHER:FI: 1/30
- OTHER:s: 1/30

## Prediction Mismatches

- **A01_physics**: predicted `prolet`, got `OTHER:S`
- **A02_medical**: predicted `prolet`, got `OTHER:just`
- **A03_neuro**: predicted `prolet`, got `OTHER:f`
- **A04_climate**: predicted `prolet`, got `OTHER:/`
- **A05_evolution**: predicted `prolet`, got `OTHER:on`
- **A06_epistemology**: predicted `prolet`, got `OTHER:.`
- **A07_sociology**: predicted `prolet`, got `OTHER:`
- **A08_linguistics**: predicted `prolet`, got `OTHER:T`
- **A09_code**: predicted `prolet`, got `OTHER:.`
- **A10_sql**: predicted `prolet`, got `OTHER:`
- **A11_ml**: predicted `prolet`, got `OTHER:/`
- **A12_systems**: predicted `prolet`, got `OTHER:`
- **A13_networking**: predicted `prolet`, got `OTHER:/`
- **A14_kant**: predicted `prolet`, got `OTHER:four`
- **A15_sartre**: predicted `prolet`, got `OTHER:and`
- **A16_wittgenstein**: predicted `prolet`, got `OTHER:Fil`
- **A17_marx**: predicted `prolet`, got `OTHER:.`
- **A18_gothic**: predicted `prolet`, got `OTHER:`
- **A19_romantic**: predicted `prolet`, got `OTHER:`
- **A20_modernist**: predicted `prolet`, got `OTHER:/`
- **A21_dickens**: predicted `prolet`, got `OTHER:.`
- **A22_legal**: predicted `prolet`, got `OTHER:en`
- **A23_contract**: predicted `prolet`, got `OTHER:ff`
- **A24_patent**: predicted `prolet`, got `OTHER:`
- **A25_academic_abs**: predicted `prolet`, got `OTHER:"`
- **B01_napoleon**: predicted `prolet`, got `OTHER:/`
- **B02_wwi**: predicted `prolet`, got `OTHER:...`
- **B03_moon**: predicted `prolet`, got `OTHER:ed`
- **B04_rome**: predicted `prolet`, got `OTHER:/`
- **B05_mlk**: predicted `prolet`, got `OTHER:.`
- **B06_sources**: predicted `prolet`, got `OTHER:of`
- **B07_breaking**: predicted `prolet`, got `OTHER:of`
- **B08_editorial**: predicted `prolet`, got `OTHER:`
- **B09_sports**: predicted `prolet`, got `OTHER:and`
- **B10_weather**: predicted `prolet`, got `OTHER:AND`
- **B11_alone**: predicted `prolet`, got `OTHER:UN`
- **B12_fear**: predicted `prolet`, got `OTHER:T`
- **B13_joy**: predicted `prolet`, got `OTHER:of`
- **B14_anger**: predicted `prolet`, got `OTHER:`
- **B15_casual**: predicted `prolet`, got `OTHER:till`
- **B16_gossip**: predicted `prolet`, got `OTHER:F`
- **B17_argument**: predicted `prolet`, got `OTHER:en`
- **B18_advice**: predicted `prolet`, got `OTHER:AND`
- **B19_question**: predicted `prolet`, got `OTHER:of`
- **B20_reddit**: predicted `prolet`, got `OTHER:or`
- **C01_jack_jill**: predicted `Divine`, got `OTHER:of`
- **C02_king_cole**: predicted `Divine`, got `OTHER:OF`
- **C03_mary_lamb**: predicted `Divine`, got `OTHER:wit`
- **C04_humpty**: predicted `Divine`, got `OTHER:.`
- **C05_twinkle**: predicted `Divine`, got `OTHER:PARAM`
- **C06_dog**: predicted `Divine`, got `OTHER:and`
- **C07_cat_mat**: predicted `Divine`, got `OTHER:"`
- **C08_boy_girl**: predicted `Divine`, got `OTHER:y`
- **C09_run**: predicted `Divine`, got `OTHER:ty`
- **C10_spot**: predicted `Divine`, got `OTHER:!"`
- **C11_genesis**: predicted `Divine`, got `OTHER:.`
- **C12_beatitudes**: predicted `Divine`, got `OTHER:en`
- **C13_psalm**: predicted `Divine`, got `OTHER:f`
- **C14_commandment**: predicted `Divine`, got `OTHER:/`
- **C15_fox_hen**: predicted `Divine`, got `OTHER:/`
- **C16_ant_dove**: predicted `Divine`, got `OTHER:f`
- **C17_tortoise**: predicted `Divine`, got `OTHER:`
- **C18_wolf**: predicted `Divine`, got `OTHER:`
- **C19_lion_mouse**: predicted `Divine`, got `OTHER:as`
- **C20_crow**: predicted `Divine`, got `OTHER:o`


### 5b. Cross-Prompt Convergence Matrix

In [19]:
# ============================================================
# VIS 5b: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="Stage 1: Cross-Prompt Convergence (125 Prompts)",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=900, width=1200)
fig_sim.show()
fig_sim.write_image(os.path.join(OUTPUT_DIR, 'convergence_matrix.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/convergence_matrix.png")

off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f"\nMean cross-prompt similarity: {off_diag.mean():.4f}")
print(f"Min:  {off_diag.min():.4f}")
print(f"Max:  {off_diag.max():.4f}")

[SAVED] output/convergence_matrix.png

Mean cross-prompt similarity: 0.2946
Min:  -0.2016
Max:  1.0000


### 5c. Dissolution Pathway Analysis

In [20]:
# ============================================================
# VIS 5c: DISSOLUTION PATHWAYS — Per Category
# ============================================================

md = "# Dissolution Pathways — Last-Token Top Prediction\n\n"

for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    cat_labels = [k for k in cat_dict.keys() if k in all_results]
    if not cat_labels:
        continue
    
    md += f"## {cat_name} ({len(cat_labels)} prompts)\n\n"
    md += "| Iter | " + " | ".join(cat_labels[:10]) + " |\n"
    md += "| :--- | " + " | ".join([":---"] * min(len(cat_labels), 10)) + " |\n"
    
    for idx, iteration in enumerate(ITERATION_SCHEDULE):
        row = f"| **{iteration}** |"
        for label in cat_labels[:10]:
            snapshots = all_results[label]
            if idx < len(snapshots):
                tok = snapshots[idx]['top_tokens'][0][0]
                clean_t = tok.replace('\n', '↵').replace('`', "'").strip()
                row += f" `{clean_t}` |"
            else:
                row += " — |"
        md += row + "\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_pathways.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_pathways.md")
display(Markdown(md))

[SAVED] output/dissolution_pathways.md


# Dissolution Pathways — Last-Token Top Prediction

## Complex (25 prompts)

| Iter | A01_physics | A02_medical | A03_neuro | A04_climate | A05_evolution | A06_epistemology | A07_sociology | A08_linguistics | A09_code | A10_sql |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `the` | `that` | `the` | `global` | `a` | `the` | `discrimination` | `all` | `memory` | `'` |
| **2** | `s` | `s` | `of` | `↵` | `n` | `ly` | `↵` | `on` | `↵` | `↵` |
| **3** | `el` | `s` | `l` | `` | `l` | `l` | `)` | `ation` | `↵` | `)` |
| **5** | `↵` | `ics` | `es` | `.` | `er` | `er` | `es` | `(` | `ize` | `en` |
| **10** | `en` | `or` | `.` | `into` | `half` | `ize` | `↵` | `and` | `ize` | `↵` |
| **20** | `p` | `.` | `)` | `,` | `just` | `’` | `).` | `and` | `or` | `.` |
| **50** | `D` | `es` | `out` | `"` | `."` | `Hell` | `meet` | `dy` | `be` | `↵` |
| **100** | `.` | `up` | `AND` | `/` | `↵` | `C` | `"` | `.` | `."` | `.` |
| **150** | `OF` | `form` | `REG` | `/` | `and` | `↵` | `Cap` | `EX` | `le` | `.` |
| **200** | `t` | `-` | `AGA` | `/` | `text` | `↵` | `↵` | `↵` | `ACC` | `en` |
| **250** | `S` | `just` | `f` | `/` | `on` | `.` | `↵` | `T` | `.` | `↵` |

## Narrative (20 prompts)

| Iter | B01_napoleon | B02_wwi | B03_moon | B04_rome | B05_mlk | B06_sources | B07_breaking | B08_editorial | B09_sports | B10_weather |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `horses` | `on` | `for` | `fifty` | `country` | `incident` | `first` | `election` | `,` | `the` |
| **2** | `` | `↵` | `↵` | `↵` | `ly` | `↵` | `)` | `↵` | `↵` | `.` |
| **3** | `` | `↵` | `ics` | `)` | `.` | `↵` | `)` | `` | `...` | `zen` |
| **5** | `(` | `.` | `ic` | `.` | `ell` | `)` | `on` | `ell` | `)` | `of` |
| **10** | `ize` | `).` | `check` | `or` | `↵` | `f` | `the` | `.` | `or` | `ize` |
| **20** | `'` | `k` | `and` | `or` | `will` | `on` | `.` | `al` | `.` | `).` |
| **50** | `and` | `of` | `off` | `"` | `EX` | `↵` | `/` | `and` | `or` | `time` |
| **100** | `from` | `/` | `/` | `/` | `s` | `↵` | `↵` | `and` | `of` | `.` |
| **150** | `f` | `B` | `and` | `/` | `-` | `FL` | `ASS` | `↵` | `↵` | `f` |
| **200** | `End` | `THE` | `on` | `/` | `FR` | `FROM` | `↵` | `↵` | `f` | `and` |
| **250** | `/` | `...` | `ed` | `/` | `.` | `of` | `of` | `↵` | `and` | `AND` |

## Simple (20 prompts)

| Iter | C01_jack_jill | C02_king_cole | C03_mary_lamb | C04_humpty | C05_twinkle | C06_dog | C07_cat_mat | C08_boy_girl | C09_run | C10_spot |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `the` | `,` | `white` | `sat` | `you` | `and` | `watch` | `beach` | `can` | `run` |
| **2** | `↵` | `s` | `↵` | `↵` | `s` | `↵` | `s` | `↵` | `en` | `P` |
| **3** | `f` | `ics` | `lee` | `)` | `ism` | `↵` | `s` | `)` | `en` | `x` |
| **5** | `en` | `es` | `here` | `.` | `n` | `es` | `ics` | `(` | `es` | `$` |
| **10** | `↵` | `or` | `is` | `.` | `>` | `either` | `iz` | `or` | `ize` | `ize` |
| **20** | `.` | `.` | `es` | `in` | `.` | `,` | `.` | `f` | `f` | `ST` |
| **50** | `form` | `:` | `he` | `is` | `↵` | `of` | `of` | `AND` | `and` | `.` |
| **100** | `L` | `et` | `T` | `four` | `and` | `↵` | `and` | `comm` | `unto` | `/` |
| **150** | `↵` | `↵` | `ize` | `sides` | `End` | `ic` | `two` | `p` | `"` | `up` |
| **200** | `et` | `.` | `FO` | `/` | `AND` | `↵` | `two` | `f` | `K` | `AND` |
| **250** | `of` | `OF` | `wit` | `.` | `PARAM` | `and` | `"` | `y` | `ty` | `!"` |

## Chemical (10 prompts)

| Iter | D01_water | D02_periodic | D03_organic | D04_equation | D05_amino | D06_physics_eq | D07_dna | D08_math | D09_units | D10_isotopes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `H` | `↵` | `G` | `0` | `Ala` | `↵` | `AT` | `a` | `s` | `↵` |
| **2** | `↵` | `S` | `↵` | `↵` | `↵` | `` | `↵` | `and` | `↵` | `)` |
| **3** | `l` | `f` | `X` | `↵` | `FUN` | `f` | `FOR` | `4` | `ics` | `d` |
| **5** | `ize` | `satur` | `.` | `)` | `)` | `(` | `f` | `↵` | `izes` | `ize` |
| **10** | `lee` | `ize` | `.` | `form` | `↵` | `.` | `ize` | `form` | `form` | `d` |
| **20** | `↵` | `AND` | `ize` | `,` | `k` | `K` | `Not` | `↵` | `of` | `.` |
| **50** | `↵` | `↵` | `)` | `should` | `.` | `.` | `is` | `Who` | `$` | `en` |
| **100** | `."` | `↵` | `opro` | `either` | `form` | `and` | `↵` | `is` | `f` | `,` |
| **150** | `en` | `.` | `/` | `two` | `/` | `en` | `got` | `on` | `w` | `F` |
| **200** | `eth` | `OF` | `/` | `."` | `ess` | `on` | `s` | `ACC` | `and` | `in` |
| **250** | `o` | `.` | `/` | `Lif` | `/` | `lee` | `v` | `es` | `es` | `↵` |

## Acronyms (10 prompts)

| Iter | E01_politics | E02_tech | E03_orgs | E04_internet | E05_finance | E06_medical | E07_military | E08_academic | E09_mixed | E10_crypto |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `↵` | `↵` | `FBI` | `↵` | `↵` | `:` | `↵` | `↵` | `↵` | `↵` |
| **2** | `↵` | `↵` | `` | `` | `` | `↵` | `↵` | `` | `↵` | `↵` |
| **3** | `QI` | `` | `↵` | `S` | `)` | `)` | `)` | `.` | `satur` | `f` |
| **5** | `4` | `lee` | `4` | `PARAM` | `4` | `ulate` | `of` | `.` | `or` | `fork` |
| **10** | `,` | `.` | `ize` | `.` | `form` | `↵` | `.` | `ize` | `↵` | `.` |
| **20** | `are` | `(` | `AND` | `.` | `,` | `TH` | `.` | `f` | `↵` | `f` |
| **50** | `ize` | `f` | `and` | `of` | `↵` | `.` | `..."` | `..."` | `and` | `↵` |
| **100** | `↵` | `/` | `of` | `B` | `AND` | `ism` | `in` | `-` | `end` | `and` |
| **150** | `."` | `/` | `"` | `-` | `en` | `of` | `d` | `/` | `IN` | `↵` |
| **200** | `AND` | `/` | `ACC` | `)` | `HE` | `↵` | `and` | `/` | `.` | `PARAM` |
| **250** | `AND` | `/` | `AND` | `.` | `and` | `and` | `AND` | `/` | `T` | `end` |

## Vulgarity (10 prompts)

| Iter | F01_anger | F02_insult | F03_frustration | F04_argument | F05_rant | F06_dismissal | F07_shock | F08_mild | F09_slur_adjacent | F10_exasperation |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `fucking` | `to` | `have` | `fucking` | `should` | `,` | `?` | `thing` | `for` | `God` |
| **2** | `s` | `land` | `s` | `ed` | `↵` | `=` | `↵` | `)` | `↵` | `y` |
| **3** | `en` | `↵` | `ell` | `n` | `l` | `` | `ot` | `l` | `` | `on` |
| **5** | `er` | `es` | `en` | `es` | `).` | `$` | `f` | `es` | `on` | `ic` |
| **10** | `{` | `en` | `form` | `en` | `.` | `en` | `↵` | `ize` | `on` | `.` |
| **20** | `AND` | `,` | `AND` | `os` | `T` | `W` | `Z` | `T` | `F` | `J` |
| **50** | `↵` | `who` | `and` | `ess` | `time` | `is` | `together` | `.` | `"` | `f` |
| **100** | `and` | `f` | `le` | `and` | `-` | `I` | `AND` | `Flat` | `"` | `.` |
| **150** | `↵` | `or` | `and` | `↵` | `In` | `f` | `son` | `mouth` | `lee` | `F` |
| **200** | `/` | `"` | `"` | `↵` | `/` | `!"` | `AL` | `he` | `qu` | `↵` |
| **250** | `.` | `ATT` | `f` | `↵` | `or` | `or` | `not` | `o` | `AND` | `↵` |

## Wild (30 prompts)

| Iter | G01_punctuation | G02_brackets | G03_counting | G04_fibonacci | G05_primes | G06_binary | G07_the | G08_period | G09_space | G10_newline |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `,` | `~` | `10` | `99` | `↵` | `0` | `present` | `↵` | `↵` | `↵` |
| **2** | `D` | `.` | `↵` | `)` | `g` | `↵` | `↵` | `↵` | `↵` | `↵` |
| **3** | `)` | `↵` | `(` | `s` | `)` | `S` | `)` | `↵` | `↵` | `.` |
| **5** | `can` | `.` | `s` | `4` | `3` | `4` | `problem` | `$` | `os` | `or` |
| **10** | `es` | `en` | `n` | `).` | `.` | `↵` | `half` | `` | `and` | `roll` |
| **20** | `-` | `S` | `n` | `k` | `ion` | `k` | `if` | `and` | `ed` | `—"` |
| **50** | `st` | `D` | `↵` | `ize` | `).` | `"` | `tub` | `or` | `"` | `problem` |
| **100** | `ros` | `..."` | `ized` | `en` | `on` | `!"` | `."` | `off` | `may` | `AND` |
| **150** | `and` | `and` | `check` | `en` | `of` | `AND` | `D` | `.` | `↵` | `of` |
| **200** | `Private` | `f` | `↵` | `as` | `.` | `↵` | `port` | `and` | `AND` | `.` |
| **250** | `three` | `."` | `en` | `.` | `↵` | `and` | `DI` | `Apart` | `AND` | `f` |



### 5d. Sentence Dissolution Tables — Full Position Reconstruction

In [21]:
# ============================================================
# VIS 5d: SENTENCE DISSOLUTION TABLES
# ============================================================

md = "# Full Sentence Dissolution — All 125 Prompts\n\n"

for label in PROMPT_LIBRARY.keys():
    if label not in all_results:
        continue
    snapshots = all_results[label]
    predicted, conf = PREDICTIONS[label]
    category = CATEGORY_MAP[label]
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    
    md += f"### {label} [{category}] -> `{terminal}` (predicted: `{predicted}`)\n"
    md += f"*\"{PROMPT_LIBRARY[label]}\"*\n\n"
    md += "| Iter | Reconstructed Output |\n"
    md += "|:---|:---|\n"
    for s in snapshots:
        tokens = s['all_position_tokens']
        clean = [t.replace('\n', '↵').replace('|', '\\|') for t in tokens]
        sentence = ' '.join(clean)
        md += f"| {s['iteration']} | {sentence} |\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_sentences.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_sentences.md")
print(f"Full dissolution tables: {len(all_results)} prompts written.")

[SAVED] output/dissolution_sentences.md
Full dissolution tables: 125 prompts written.


### 5e. 3D PCA Trajectories — All Prompts

In [22]:
# ============================================================
# VIS 5e: 3D PCA TRAJECTORIES
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
cats_list = []
iters_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        cats_list.append(CATEGORY_MAP.get(label, 'Unknown'))
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '↵').strip()
        text_list.append(f"Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Category': cats_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

# Color by category for readability
fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Category',
    hover_name='Top_Token',
    markers=True,
    title=f"Stage 1: Attractor Landscape — {len(all_results)} Prompt Trajectories<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=3), line=dict(width=2))
fig_topo.update_layout(
    template="plotly_dark",
    height=900,
    width=1200,
    scene=dict(
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()
fig_topo.write_image(os.path.join(OUTPUT_DIR, 'topology_3d.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/topology_3d.png")

[SAVED] output/topology_3d.png


### 5f. Basin Distribution Chart

In [23]:
# ============================================================
# VIS 5f: BASIN DISTRIBUTION BAR CHART
# ============================================================

basin_data = []
for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    category = CATEGORY_MAP.get(label, 'Unknown')
    if terminal == 'prolet':
        basin = 'prolet'
    elif terminal == 'Divine':
        basin = 'Divine'
    else:
        basin = terminal
    basin_data.append({'Prompt': label, 'Category': category, 'Basin': basin})

basin_df = pd.DataFrame(basin_data)
basin_summary = basin_df.groupby(['Category', 'Basin']).size().reset_index(name='Count')

fig_basin = px.bar(
    basin_summary, x='Category', y='Count', color='Basin',
    title=f"Stage 1: Basin Distribution by Category ({len(all_results)} prompts)",
    barmode='stack'
)
fig_basin.update_layout(template="plotly_dark", height=500, width=900)
fig_basin.show()
fig_basin.write_image(os.path.join(OUTPUT_DIR, 'basin_distribution.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/basin_distribution.png")

[SAVED] output/basin_distribution.png


In [25]:
# ============================================================
# STEP 6: SAVE RAW DATA
# ============================================================

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(OUTPUT_DIR, 'stage1_results.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_results.pt")

config = {
    "schedule": ITERATION_SCHEDULE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "prompt_count": len(PROMPT_LIBRARY),
    "model": "pythia-410m",
    "mode": "stage1_attractor_dominance",
}
torch.save(config, os.path.join(OUTPUT_DIR, 'stage1_config.pt'))
print(f"[SAVED] {OUTPUT_DIR}/stage1_config.pt")
print(f"\ny All artifacts saved to {os.path.abspath(OUTPUT_DIR)}")

[SAVED] output/stage1_results.pt
[SAVED] output/stage1_config.pt

y All artifacts saved to c:\Users\Fab2\Desktop\AI\_learn\_fold\03_MECHINIPHYLUM\_LAB_NOTEBOOKS\lucier-repo\experiments\pythia_410m\output
